In [2]:
import pandas as pd
from ortools.sat.python import cp_model

# Load dataset
df = pd.read_csv("../data/kmrl_enhanced_dataset.csv")

# Parameters (Buffer settings)
BUFFER_DAYS = 7
BUFFER_KM = 5000  # example buffer for BrakePad / Bogie wear

def optimize_schedule(data):
    model = cp_model.CpModel()
    decisions = {}

    for idx, row in data.iterrows():
        service = model.NewBoolVar(f"service_{idx}")
        standby = model.NewBoolVar(f"standby_{idx}")
        ibl = model.NewBoolVar(f"ibl_{idx}")
        model.Add(service + standby + ibl == 1)  # only one decision

        # Hard constraints examples:
        if row["BrakePad_KM_Since_Change"] >= (20000 - BUFFER_KM):
            model.Add(ibl == 1)
        elif row["Bogie_KM_Since_Service"] >= (30000 - BUFFER_KM):
            model.Add(ibl == 1)
        else:
            # If not risky, allow service or standby
            model.Add(standby + service >= 1)

        decisions[idx] = (service, standby, ibl)

    # Objective: balance service usage (example)
    model.Maximize(sum(dec[0] for dec in decisions.values()))

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 10
    solver.Solve(model)

    results = []
    for idx, row in data.iterrows():
        s, st, ib = decisions[idx]
        if solver.Value(s):
            choice = "Service"
        elif solver.Value(st):
            choice = "Standby"
        else:
            choice = "IBL"
        results.append({"Train_ID": row["Train_ID"], "OptimizedDecision": choice})

    results_df = pd.DataFrame(results)
    results_df.to_csv("../data/final_schedule.csv", index=False)
    print(" Final schedule saved: ../data/final_schedule.csv")
    return results_df

# Run optimization
optimized = optimize_schedule(df)
print(optimized.head())


 Final schedule saved: ../data/final_schedule.csv
   Train_ID OptimizedDecision
0  Train_01           Service
1  Train_02           Service
2  Train_03           Service
3  Train_04           Service
4  Train_05           Service
